In [16]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import gc

# 1. DEFINE A SCHEMA
# Loading 'event_name' as a category immediately saves ~80% RAM
dtypes = {
    'id': 'str',
    'event_name': 'category'
}

# 2. LOAD INDIVIDUALLY AND CLEAN
def load_and_clean(file_path):
    # Load with dtypes to save memory instantly
    df = pd.read_csv(file_path, dtype=dtypes)
    
    # Process datetime immediately to free up string memory
    df['event_timestamp'] = pd.to_datetime(df['event_timestamp'], utc=True, errors='coerce')
    
    # Process IDs without creating intermediate lists
    df['customer_id'] = df['id'].str.extract(r'^([^\s]+)', expand=False)
    
    # --- ADDED LOGIC: SORT, DEDUPLICATE, AND ADD STEPS ---
    # 1. Sort chronologically to ensure cumcount() applies steps in the right order
    df = df.sort_values(by=['id', 'event_timestamp'])
    
    # 2. Remove duplicates
    duplicate_mask = df.duplicated(subset=['id', 'event_name', 'event_timestamp'], keep='first')
    df = df[~duplicate_mask].copy()
    
    # 3. Calculate journey steps securely
    df['journey_step'] = df.groupby('id').cumcount() + 1
    df.drop(columns=['journey_steps_until_end'], inplace=True)
    
    gc.collect() # Clean artifacts from extraction
    return df

# Process one at a time
df_train2 = load_and_clean('data/dat_train2.csv')
gc.collect()

df_test2 = load_and_clean('data/open_journeys2.csv')
gc.collect()

# --- ISOLATE TEST DATA TO PREVENT LEAKAGE ---
test_ids = set(df_test2['id'].unique())

# Identify the true 'Present Day' across all data before we start dropping rows
global_max_date = max(df_train2['event_timestamp'].max(), df_test2['event_timestamp'].max())

# Drop all test IDs from the training dataframe
df_train2 = df_train2[~df_train2['id'].isin(test_ids)].copy()
gc.collect()


# 3. IDENTIFY SUCCESS 
success_ids = df_train2.loc[df_train2['event_name'] == 'order_shipped', 'id'].unique()
success_set = set(success_ids) # O(1) lookup speed

print(f"Setup Complete. Training rows after isolating test data: {len(df_train2)}")
gc.collect() 

# --- STEP 2: VECTORIZED LABELING (FIXED FOR MEMORY) ---

# 1. Get the last event timestamp ONLY for unique IDs
journey_end_times = df_train2.groupby('id')['event_timestamp'].max()

# 2. Create a dedicated Labels DataFrame
train_labels = pd.DataFrame({'id': df_train2['id'].unique()})

# 3. Map the end times and calculate days since last event using the global max date
train_labels['last_event'] = train_labels['id'].map(journey_end_times)
train_labels['days_since_last'] = (global_max_date - train_labels['last_event'])

# 4. Vectorized Logical Conditions
is_success = train_labels['id'].isin(success_set)
is_lapsed = train_labels['days_since_last'] >= pd.Timedelta(days=60)

# 5. Apply Labels (1 = Success, 0 = Lapsed, -1 = Still Active)
train_labels['label'] = np.select(
    [is_success, is_lapsed], 
    [1, 0], 
    default=-1
)

# 6. Final Training Filter
train_labels = train_labels[train_labels['label'] != -1][['id', 'label']].copy()

print(f"Labeling complete. Found {len(train_labels)} valid journeys for training.")
print(f"Success Rate: {train_labels['label'].mean():.2%}")

Setup Complete. Training rows after isolating test data: 51555067
Labeling complete. Found 1450231 valid journeys for training.
Success Rate: 20.90%


In [27]:
# --- SUB-SAMPLING & UNIFORM TIME CUTOFF ---
import random
import numpy as np
import pandas as pd

# 1. Downsample successful journeys to hit target Kaggle balance
target_success_rate = 0.05 

current_successes = train_labels[train_labels['label'] == 1]
current_failures = train_labels[train_labels['label'] == 0]

desired_success_count = int(len(current_failures) * (target_success_rate / (1.0 - target_success_rate)))

if len(current_successes) > desired_success_count:
    kept_successes = current_successes.sample(n=desired_success_count, random_state=42)
    train_labels_balanced = pd.concat([current_failures, kept_successes])
else:
    train_labels_balanced = train_labels.copy()

balanced_ids = set(train_labels_balanced['id'])
df_train_balanced = df_train2[df_train2['id'].isin(balanced_ids)].copy()

# 2. Uniform Time Cutoff 
journey_times = df_train_balanced.groupby('id')['event_timestamp'].agg(['min', 'max']).reset_index()
journey_times['time_diff'] = (journey_times['max'] - journey_times['min']).dt.total_seconds()

np.random.seed(42)
journey_times['random_offset'] = journey_times['time_diff'] * np.random.uniform(0, 1, len(journey_times))
journey_times['cutoff_time'] = journey_times['min'] + pd.to_timedelta(journey_times['random_offset'], unit='s')

df_train_cut = df_train_balanced.merge(journey_times[['id', 'cutoff_time']], on='id')
df_train_truncated = df_train_cut[df_train_cut['event_timestamp'] <= df_train_cut['cutoff_time']].copy()

df_train_truncated.drop(columns=['cutoff_time'], inplace=True)

# CRITICAL: Update the labels reference so Cell 18 merges correctly
train_labels = train_labels_balanced 

print(f"Balancing and Truncation complete. Rows remaining: {len(df_train_truncated)}")

Balancing and Truncation complete. Rows remaining: 26658807


In [28]:
def extract_elite_features_v10(df_input):
    df = df_input.copy()
    
    # 1. Base Time Features
    df['reference_time'] = df.groupby('id')['event_timestamp'].transform('max')
    df['journey_start'] = df.groupby('id')['event_timestamp'].transform('min')
    df['time_since_began'] = (df['reference_time'] - df['journey_start']).dt.total_seconds() / 3600.0
    
    df['prev_time'] = df.groupby('id')['event_timestamp'].shift(1)
    df['time_between_actions'] = (df['event_timestamp'] - df['prev_time']).dt.total_seconds() / 3600.0
    
    # 2. Filter for strictly "User Actions"
    non_user_actions = [1, 20, 21]
    df['is_user_action'] = ~df['ed_id'].isin(non_user_actions)
    
    # 3. Milestone Mapping
    milestones = {
        'm2_order': [7, 18],
        'm3_account_act': [29],
        'm4_downpayment': [8],
        'm5_downpayment_cleared': [27]
    }
    
    for m_name, m_ids in milestones.items():
        df[m_name] = df['ed_id'].isin(m_ids).astype(int)
        time_to_m = (df['event_timestamp'] - df['journey_start']).dt.total_seconds() / 3600.0
        df[f'time_to_{m_name}'] = np.where(df[m_name] == 1, time_to_m, np.nan)
        
    # 4. Core Aggregations (REMOVED CUSTOM FUNCTIONS)
    agg_dict = {
        'event_name': ['first', 'last', 'count'],
        'time_since_began': 'max',
        'time_between_actions': ['max'], # Only keeping the fast 'max' here
    }
    
    for m_name in milestones.keys():
        agg_dict[m_name] = 'max' 
        agg_dict[f'time_to_{m_name}'] = 'min' 
        
    features = df.groupby('id').agg(agg_dict)
    features.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in features.columns.values]
    features = features.reset_index()
    
    # --- NEW: FAST VECTORIZED QUANTILES ---
    # This runs the quantiles through the optimized C-backend all at once
    quantiles = df.groupby('id')['time_between_actions'].quantile([0.25, 0.5, 0.75, 0.85, 0.95]).unstack()
    quantiles.columns = [f'time_between_actions_q{int(q*100)}' for q in quantiles.columns]
    features = features.merge(quantiles, on='id', how='left')
    
    # 5. Time since last user action
    user_actions_df = df[df['is_user_action']].groupby('id')['event_timestamp'].max().reset_index()
    user_actions_df.columns = ['id', 'last_user_action_time']
    ref_times = df.groupby('id')['reference_time'].first().reset_index()
    user_actions_df = user_actions_df.merge(ref_times, on='id')
    user_actions_df['time_since_last_user_action'] = (user_actions_df['reference_time'] - user_actions_df['last_user_action_time']).dt.total_seconds() / 3600.0
    
    features = features.merge(user_actions_df[['id', 'time_since_last_user_action']], on='id', how='left')
    
    # Ensure categorical types for XGBoost
    features['event_name_first'] = features['event_name_first'].astype('category')
    features['event_name_last'] = features['event_name_last'].astype('category')
    
    return features

# --- EXECUTION ---
print("Flattening Truncated Training Data with Milestone Features...")
X_train_full = extract_elite_features_v10(df_train_truncated)
df_final_train = X_train_full.merge(train_labels, on='id', how='inner')

print("Flattening Test Data...")
X_test_final = extract_elite_features_v10(df_test2)

Flattening Truncated Training Data with Milestone Features...
Flattening Test Data...


In [34]:
import xgboost as xgb

# 1. Feature Selection
drop_cols = ['id', 'label']
selected_features = [c for c in df_final_train.columns if c not in drop_cols]

# 2. Setup Static Model
X = df_final_train[selected_features]
y = df_final_train['label']

# --- UPDATED HYPERPARAMETERS TO COMBAT FEATURE DOMINANCE ---
params = {
    'n_estimators': 100,
    'max_depth': 4,               # Decreased from 6 to prevent hyper-focusing on extreme edge cases
    'colsample_bytree': 0.4,      # Decreased from 0.8 to hide the dominant feature from 60% of the trees
    'reg_lambda': 20,             # Added L2 regularization to penalize overconfident weights
    'learning_rate': 0.05,
    'objective': 'binary:logistic',
    'random_state': 42,
    'n_jobs': -1,
    'enable_categorical': True 
}

print("Initializing and Training XGBoost with Anti-Dominance Constraints...")
final_model = xgb.XGBClassifier(**params)
final_model.fit(X, y)

print("Training complete!")

Initializing and Training XGBoost with Anti-Dominance Constraints...
Training complete!


In [35]:
import numpy as np

# 1. Generate raw probabilities from the balanced model
test_raw = final_model.predict_proba(X_test_final[selected_features])[:, 1]

# 2. Apply the clipping ceiling to prevent massive Brier score penalties
final_probs = np.clip(test_raw, 0, 0.12)

# Print a quick sanity check
print(f"Mean predicted prob before saving: {final_probs.mean():.4f}")
print(f"Max prob: {final_probs.max():.4f}")

Mean predicted prob before saving: 0.0580
Max prob: 0.1200


In [36]:
df_kaggle_template = pd.read_csv('open_journeys2_flattened_all0.csv')

In [37]:
final_test_probs = final_probs
# 2. Create a lookup Series from your predictions
# We use the 'id' from X_test_final as the index for a quick lookup
preds_series = pd.Series(final_test_probs, index=X_test_final['id'].astype(str))

# 3. Map predictions to the template
# We use .map() to match the 'id' column in the template to our predictions
# .fillna(0.015) is a safety net for any IDs that had 0 events in the test log
df_kaggle_template['order_shipped'] = df_kaggle_template['id'].map(preds_series).fillna(0.015)

# 4. Final Verification
print(f"Submission Mean: {df_kaggle_template['order_shipped'].mean():.4f}")
print(f"Max Prob: {df_kaggle_template['order_shipped'].max():.4f}")
print(f"Min Prob: {df_kaggle_template['order_shipped'].min():.4f}")

# 5. Save the file
submission_name = 'xgboost_comp2_v1.csv'
df_kaggle_template[['id', 'order_shipped']].to_csv(submission_name, index=False)

print(f"\nSuccessfully saved to {submission_name}")

Submission Mean: 0.0580
Max Prob: 0.1200
Min Prob: 0.0031

Successfully saved to xgboost_comp2_v1.csv
